In [ ]:
# KALMAN OPEN REVALIDATION DERIVED REBUILD v2.0 — ONE CELL
# Research-only. Recomputes price-derived fields for complete IEX rows.
# It deliberately does NOT invent/overwrite frozen OPEN_* policy semantics.

from google.colab import drive
drive.mount("/content/drive",force_remount=False)
from pathlib import Path
from datetime import datetime, timezone
import json, shutil
import numpy as np, pandas as pd

ROOT=Path("/content/drive/MyDrive/US_ETF/model_lab_v1/results/open_revalidation_v1")
AUDIT=ROOT/"open_revalidation_trade_audit.parquet"
assert AUDIT.exists(), AUDIT
d=pd.read_parquet(AUDIT).copy()
price=["entry_price_iex","fixed4_exit_price_iex","prev_close_price_iex","open_0_price_iex","open_5_price_iex","open_15_price_iex"]
for c in price: d[c]=pd.to_numeric(d[c],errors="coerce")
complete=d[price].notna().all(axis=1)

# infer return direction from existing 20 reconstructed rows instead of assuming long/short.
known=d["reconstructed_fixed4_raw_return"].notna() & d["entry_price_iex"].notna() & d["fixed4_exit_price_iex"].notna()
long_formula=d["fixed4_exit_price_iex"]/d["entry_price_iex"]-1
short_formula=d["entry_price_iex"]/d["fixed4_exit_price_iex"]-1
err_long=np.nanmedian(np.abs(d.loc[known,"reconstructed_fixed4_raw_return"]-long_formula.loc[known])) if known.any() else np.inf
err_short=np.nanmedian(np.abs(d.loc[known,"reconstructed_fixed4_raw_return"]-short_formula.loc[known])) if known.any() else np.inf
direction="LONG" if err_long<=err_short else "SHORT"
sgn=1.0 if direction=="LONG" else -1.0
print("[CONTRACT INFERENCE] known=",int(known.sum()),"direction=",direction,"median_err_long=",err_long,"median_err_short=",err_short)
if not known.any() or min(err_long,err_short)>1e-8:
    raise RuntimeError("Could not reproduce existing reconstructed_fixed4_raw_return exactly enough; stop rather than invent semantics.")

def ret(exitp,entryp):
    return exitp/entryp-1 if direction=="LONG" else entryp/exitp-1

# Price-derived fields: overwrite only on complete rows, using explicit price ratios.
idx=complete
d.loc[idx,"position_return_prev_close"]=ret(d.loc[idx,"prev_close_price_iex"],d.loc[idx,"entry_price_iex"])
d.loc[idx,"position_return_open"]=ret(d.loc[idx,"open_0_price_iex"],d.loc[idx,"entry_price_iex"])
d.loc[idx,"position_return_5m"]=ret(d.loc[idx,"open_5_price_iex"],d.loc[idx,"entry_price_iex"])
d.loc[idx,"position_return_15m"]=ret(d.loc[idx,"open_15_price_iex"],d.loc[idx,"entry_price_iex"])
d.loc[idx,"overnight_gap_return"]=d.loc[idx,"open_0_price_iex"]/d.loc[idx,"prev_close_price_iex"]-1
d.loc[idx,"open_momentum_5m"]=d.loc[idx,"open_5_price_iex"]/d.loc[idx,"open_0_price_iex"]-1
d.loc[idx,"open_momentum_15m"]=d.loc[idx,"open_15_price_iex"]/d.loc[idx,"open_0_price_iex"]-1
# Giveback definition is validated against legacy known rows below; candidate formulas tested.
cands={
 "posprev_minus_pos5": d["position_return_prev_close"]-d["position_return_5m"],
 "pos5_minus_posprev": d["position_return_5m"]-d["position_return_prev_close"],
 "prev_to_5_raw": d["open_5_price_iex"]/d["prev_close_price_iex"]-1,
}
target=d["giveback_prev_close_to_5m"]
errs={k:(np.nanmedian(np.abs(target[known]-v[known])) if target[known].notna().any() else np.inf) for k,v in cands.items()}
gb=min(errs,key=errs.get)
print("[GIVEBACK INFERENCE]",gb,errs)
if not np.isfinite(errs[gb]) or errs[gb]>1e-8:
    raise RuntimeError("Could not reproduce legacy giveback field; stop.")
d.loc[idx,"giveback_prev_close_to_5m"]=cands[gb].loc[idx]
d.loc[idx,"reconstructed_fixed4_raw_return"]=ret(d.loc[idx,"fixed4_exit_price_iex"],d.loc[idx,"entry_price_iex"])

# net-return contract: infer exact additive cost convention from existing known rows.
cost=pd.to_numeric(d["cost_proxy"],errors="coerce")
net=pd.to_numeric(d["reconstructed_fixed4_net_return"],errors="coerce")
raw=pd.to_numeric(d["reconstructed_fixed4_raw_return"],errors="coerce")
net_known=known & net.notna() & cost.notna()
forms={
 "raw_minus_cost": raw-cost,
 "raw_plus_cost": raw+cost,
 "raw": raw
}
nerrs={k:(np.nanmedian(np.abs(net[net_known]-v[net_known])) if net_known.any() else np.inf) for k,v in forms.items()}
nf=min(nerrs,key=nerrs.get)
print("[NET CONTRACT]",nf,nerrs)
if not np.isfinite(nerrs[nf]) or nerrs[nf]>1e-8:
    raise RuntimeError("Could not reproduce legacy reconstructed_fixed4_net_return; stop.")
d.loc[idx,"reconstructed_fixed4_net_return"]=forms[nf].loc[idx]

# legacy reproduction audit for all derived fields that existed in the original ready subset
derived=["position_return_prev_close","position_return_open","position_return_5m","position_return_15m","overnight_gap_return","open_momentum_5m","open_momentum_15m","giveback_prev_close_to_5m","reconstructed_fixed4_raw_return","reconstructed_fixed4_net_return"]
print("\n[DERIVED COMPLETENESS]")
print(d[derived].isna().sum().to_string())

# Do NOT treat OPEN exit_price NaN as missing when trigger=False. Do NOT overwrite policy columns.
trigger_cols=[c for c in d if c.startswith("OPEN_") and c.endswith("__trigger")]
print("\n[OPEN POLICY STATUS — PRESERVED, NOT RECOMPUTED]")
for tc in trigger_cols:
    base=tc[:-9]; ec=base+"__exit_price"; nc=base+"__net_return"
    trig=d[tc].fillna(False).astype(bool)
    print(base,"triggered=",int(trig.sum()),"exit_price_when_trigger_missing=",int(d.loc[trig,ec].isna().sum()) if ec in d else "NA","net_missing=",int(d[nc].isna().sum()) if nc in d else "NA")

# Atomic write only if legacy contracts reproduced and completeness materially improves.
before=int(pd.read_parquet(AUDIT)["reconstructed_fixed4_net_return"].notna().sum())
after=int(d["reconstructed_fixed4_net_return"].notna().sum())
stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
backup=ROOT/f"open_revalidation_trade_audit.pre_derived_v2_0_{stamp}.parquet"
candidate=ROOT/"open_revalidation_trade_audit.derived_v2_0_candidate.parquet"
d.to_parquet(candidate,index=False)
if after>before:
    shutil.copy2(AUDIT,backup)
    candidate.replace(AUDIT)
    action="REPLACED"
else:
    action="CANDIDATE_ONLY"
report={"schema":"kalman-open-revalidation-derived-rebuild-v2.0","research_only":True,"production_changed":False,"live_trading":False,"neon_write":False,"rows":len(d),"price_complete":int(complete.sum()),"direction":direction,"giveback_contract":gb,"net_contract":nf,"derived_before":before,"derived_after":after,"action":action,"backup":str(backup) if action=="REPLACED" else None,"audit":str(AUDIT)}
(ROOT/"derived_rebuild_v2_0_report.json").write_text(json.dumps(report,indent=2))
print("\n[FINAL]\n",json.dumps(report,indent=2))
print("\nNEXT: rerun Frozen Validation only after derived_after is ~888. OPEN_* frozen semantics remain a separate gate.")
